# F1-StratML: Real-World EDA for the V1 Scope

This notebook now uses **real FastF1 API data** instead of synthetic simulation. It is scoped to the project v1 subset:

- Circuits: **Bahrain, Britain, Spain, Italy, Monaco, Singapore**
- Constructors: **Red Bull, Mercedes, Ferrari, McLaren**

The notebook is intentionally exploratory and plotting-focused. It does **not** change the repository code; it only fetches, cleans, engineers, and visualizes real race-session data inside the notebook.

## 0. Setup and Scope

The cells below fetch real FastF1 race data, derive the main strategy features used by the project, and generate EDA plots for the selected circuits and constructors.

In [ ]:
import logging
from pathlib import Path

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, kruskal, linregress
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s - %(message)s')
LOGGER = logging.getLogger('eda')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

SEASONS = [2018, 2019, 2020, 2021, 2022, 2023]
SELECTED_CIRCUITS = ['Bahrain', 'Britain', 'Spain', 'Italy', 'Monaco', 'Singapore']
SELECTED_CONSTRUCTORS = ['Red Bull', 'Mercedes', 'Ferrari', 'McLaren']
CACHE_DIR = Path('./f1_cache')
OUTPUT_DIR = Path('./processed')
FUEL_KG_START = 110.0
FUEL_CONSUMPTION_PER_LAP = 2.3
FUEL_LAP_TIME_EFFECT = 0.1495
UNDERCUT_GAP_S = 3.0
UNDERCUT_LOOKBACK_LAPS = 2
SC_CODES = {'4', '5'}
VSC_CODES = {'6', '7'}

EVENT_ALIASES = {
    'silverstone': 'Britain',
    'great britain': 'Britain',
    'britain': 'Britain',
    'barcelona': 'Spain',
    'spanish': 'Spain',
    'spain': 'Spain',
    'monza': 'Italy',
    'italian': 'Italy',
    'italy': 'Italy',
    'bahrain': 'Bahrain',
    'monaco': 'Monaco',
    'singapore': 'Singapore',
}

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))
print('FastF1 cache enabled at', CACHE_DIR.resolve())
print('Circuits:', SELECTED_CIRCUITS)
print('Constructors:', SELECTED_CONSTRUCTORS)


In [ ]:
def canonical_event_name(raw_name: str) -> str:
    normalized = str(raw_name).strip().lower()
    for alias, canonical in EVENT_ALIASES.items():
        if alias in normalized:
            return canonical
    return str(raw_name)


def team_matches(team_name: str, reference: str) -> bool:
    team_norm = str(team_name).lower()
    ref_norm = str(reference).lower()
    return team_norm in ref_norm or ref_norm in team_norm


def selected_team_mask(team_series: pd.Series) -> pd.Series:
    return team_series.fillna('').astype(str).apply(
        lambda team: any(team_matches(team, ref) for ref in SELECTED_CONSTRUCTORS)
    )


def get_event_name(row: pd.Series) -> str:
    for column in ('EventName', 'OfficialEventName', 'Country', 'Location'):
        if column in row and pd.notna(row[column]):
            return canonical_event_name(row[column])
    raise KeyError('No event-like column found in schedule row')


def merge_weather(laps: pd.DataFrame, weather: pd.DataFrame) -> pd.DataFrame:
    if laps.empty or weather.empty or 'Time' not in laps.columns or 'Time' not in weather.columns:
        return laps.copy()
    return pd.merge_asof(
        laps.sort_values('Time').copy(),
        weather.sort_values('Time').copy(),
        on='Time',
        direction='nearest',
    )


def load_selected_race_laps(seasons: list[int]) -> pd.DataFrame:
    frames = []
    for year in seasons:
        LOGGER.info('Loading schedule for %s', year)
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        for _, event_row in schedule.iterrows():
            event_name = get_event_name(event_row)
            if event_name not in SELECTED_CIRCUITS:
                continue
            LOGGER.info('Loading %s %s race session', year, event_name)
            session = fastf1.get_session(year, event_name, 'R')
            session.load(telemetry=False, weather=True, messages=False)
            laps = session.laps.pick_accurate().copy()
            if laps.empty:
                continue
            laps = merge_weather(laps, session.weather_data.copy())
            laps['season'] = year
            laps['event_name'] = event_name
            laps['session_type'] = 'R'
            frames.append(laps)
        LOGGER.info('Finished %s', year)
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()


In [ ]:
raw_laps = load_selected_race_laps(SEASONS)
print('Raw shape:', raw_laps.shape)
raw_laps[['season', 'event_name', 'Driver', 'Team', 'LapNumber', 'Compound']].head()


## 1. Real-World Feature Engineering

In [ ]:
eda_df = raw_laps.copy()
eda_df = eda_df[selected_team_mask(eda_df['Team'])].copy()
eda_df['driver_stint_id'] = (
    eda_df['season'].astype(str) + '_' + eda_df['event_name'].astype(str) + '_'
    + eda_df['Driver'].astype(str) + '_' + eda_df['Stint'].astype(str)
)
eda_df = eda_df.sort_values(['driver_stint_id', 'LapNumber']).copy()
eda_df['LapTime_s'] = eda_df['LapTime'].dt.total_seconds()
eda_df['fuel_load_kg'] = (FUEL_KG_START - eda_df['LapNumber'] * FUEL_CONSUMPTION_PER_LAP).clip(lower=0.0)
eda_df['tire_age_laps'] = pd.to_numeric(eda_df['TyreLife'], errors='coerce')
eda_df['tire_age_sq'] = eda_df['tire_age_laps'] ** 2
eda_df['_lap_in_stint'] = eda_df.groupby('driver_stint_id').cumcount() + 1
eda_df['_corrected'] = eda_df['LapTime_s'] - eda_df['_lap_in_stint'] * FUEL_LAP_TIME_EFFECT

def stint_baseline(group: pd.DataFrame) -> pd.Series:
    ref = group.iloc[1:5] if len(group) >= 4 else group
    baseline = ref['_corrected'].median()
    return pd.Series(np.repeat(baseline, len(group)), index=group.index)

eda_df['_baseline'] = eda_df.groupby('driver_stint_id', group_keys=False).apply(stint_baseline, include_groups=False)
eda_df['lap_time_delta_s'] = eda_df['_corrected'] - eda_df['_baseline']
eda_df['deg_velocity'] = eda_df.groupby('driver_stint_id')['lap_time_delta_s'].diff().fillna(0.0)
eda_df['deg_acceleration'] = eda_df.groupby('driver_stint_id')['deg_velocity'].diff().fillna(0.0)

eda_df['TrackStatus'] = eda_df['TrackStatus'].astype(str)
eda_df['safety_car_flag'] = eda_df['TrackStatus'].isin(SC_CODES).astype(int)
eda_df['vsc_flag'] = eda_df['TrackStatus'].isin(VSC_CODES).astype(int)
eda_df['sc_or_vsc_flag'] = (eda_df['safety_car_flag'] | eda_df['vsc_flag']).astype(int)

eda_df['_pos_num'] = pd.to_numeric(eda_df['Position'], errors='coerce')
eda_df['_gap_leader_s'] = pd.to_numeric(eda_df['GapToLeader'], errors='coerce')
lap_keys = ['season', 'event_name', 'session_type', 'LapNumber']
eda_df = eda_df.sort_values(lap_keys + ['_pos_num', 'Driver'], kind='mergesort').copy()
eda_df['gap_ahead_s'] = eda_df.groupby(lap_keys)['_gap_leader_s'].diff().clip(lower=0.0).fillna(0.0)
eda_df.loc[eda_df.groupby(lap_keys).cumcount() == 0, 'gap_ahead_s'] = 0.0
eda_df['gap_ahead_trend'] = eda_df.groupby('driver_stint_id')['gap_ahead_s'].diff(3).fillna(0.0)

eda_df['undercut_threat'] = 0
for session_key, session_df in eda_df.groupby(['season', 'event_name', 'session_type']):
    session_index = session_df.index
    temp = session_df.copy()
    temp['undercut_threat'] = 0
    for lap_n in sorted(temp['LapNumber'].dropna().unique()):
        recent_pitters = temp[
            temp['LapNumber'].isin(range(int(lap_n) - UNDERCUT_LOOKBACK_LAPS, int(lap_n)))
            & temp['PitInTime'].notna()
        ]['Driver'].unique()
        if len(recent_pitters) == 0:
            continue
        current_lap = temp[temp['LapNumber'] == lap_n]
        for idx, row in current_lap.iterrows():
            my_gap = row.get('_gap_leader_s', np.nan)
            if pd.isna(my_gap):
                continue
            rivals = current_lap[(current_lap['Driver'].isin(recent_pitters)) & (current_lap['Driver'] != row['Driver'])]
            threatening = rivals['_gap_leader_s'].apply(lambda rival_gap: pd.notna(rival_gap) and rival_gap < my_gap and (my_gap - rival_gap) < UNDERCUT_GAP_S)
            if threatening.any():
                temp.loc[idx, 'undercut_threat'] = 1
    eda_df.loc[session_index, 'undercut_threat'] = temp['undercut_threat']

eda_df['pit_label'] = 0
eda_df['loss_weight'] = 1.0
eda_df.loc[eda_df['sc_or_vsc_flag'] == 1, 'loss_weight'] = 0.0
for driver, driver_df in eda_df.groupby('Driver'):
    pit_laps = driver_df[driver_df['PitInTime'].notna()]['LapNumber'].values
    for pit_lap in pit_laps:
        eda_df.loc[(eda_df['Driver'] == driver) & (eda_df['LapNumber'] == pit_lap), 'pit_label'] = 2
        for offset in [1, 2, 3]:
            warn_lap = pit_lap - offset
            mask = (eda_df['Driver'] == driver) & (eda_df['LapNumber'] == warn_lap) & (eda_df['pit_label'] == 0)
            eda_df.loc[mask, 'pit_label'] = 1

eda_df['circuit_type'] = eda_df['event_name'].map({'Monaco': 'street', 'Singapore': 'street'}).fillna('permanent')
eda_df['lap_norm'] = eda_df['LapNumber'] / eda_df.groupby(['season', 'event_name'])['LapNumber'].transform('max')

eda_df = eda_df[eda_df['LapNumber'] > 1].copy()
eda_df = eda_df[eda_df['LapTime_s'].notna() & (eda_df['LapTime_s'] > 0)].copy()
eda_df = eda_df[~eda_df['Compound'].isin(['INTERMEDIATE', 'WET'])].copy()

keep_cols = [
    'season', 'event_name', 'session_type', 'Driver', 'Team', 'Stint', 'LapNumber', 'Compound',
    'Position', 'LapTime_s', 'lap_time_delta_s', 'deg_velocity', 'deg_acceleration', 'fuel_load_kg',
    'tire_age_laps', 'tire_age_sq', 'gap_ahead_s', 'gap_ahead_trend', 'undercut_threat',
    'safety_car_flag', 'vsc_flag', 'sc_or_vsc_flag', 'pit_label', 'loss_weight', 'driver_stint_id',
    'circuit_type', 'lap_norm', 'TrackTemp', 'AirTemp', 'Humidity', 'PitInTime', 'PitOutTime'
]
eda_df = eda_df[keep_cols].copy()
print('EDA dataframe shape:', eda_df.shape)
eda_df.head()


## 2. Dataset Overview and Label Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Real-World Dataset Overview for the V1 Scope', fontsize=16, fontweight='bold')

label_counts = eda_df['pit_label'].value_counts().sort_index()
axes[0, 0].bar(['stay_out', 'pit_soon', 'pit_now'], label_counts.values, color=['#4A90D9', '#F5A623', '#E10600'])
axes[0, 0].set_title('Pit Label Distribution')

compound_counts = eda_df.groupby('Compound')['driver_stint_id'].nunique().reindex(['SOFT', 'MEDIUM', 'HARD']).fillna(0)
axes[0, 1].pie(compound_counts.values, labels=compound_counts.index, autopct='%1.1f%%', colors=['#FF4444', '#FFD700', '#CCCCCC'])
axes[0, 1].set_title('Stints by Compound')

circuit_counts = eda_df.groupby('event_name')['driver_stint_id'].nunique().sort_values()
axes[0, 2].barh(circuit_counts.index, circuit_counts.values, color='#7B68EE')
axes[0, 2].set_title('Stints by Circuit')

team_counts = eda_df.groupby('Team')['driver_stint_id'].nunique().sort_values()
axes[1, 0].barh(team_counts.index, team_counts.values, color='#2E8B57')
axes[1, 0].set_title('Stints by Constructor')

for compound, color in [('SOFT', '#FF4444'), ('MEDIUM', '#FFD700'), ('HARD', '#CCCCCC')]:
    lengths = eda_df[eda_df['Compound'] == compound].groupby('driver_stint_id')['tire_age_laps'].max().dropna()
    axes[1, 1].hist(lengths, bins=20, alpha=0.6, label=compound, color=color)
axes[1, 1].set_title('Stint Length Distribution')
axes[1, 1].legend()

season_counts = eda_df.groupby('season')['driver_stint_id'].nunique()
axes[1, 2].bar(season_counts.index.astype(str), season_counts.values, color='#4A90D9')
axes[1, 2].set_title('Stints by Season')

plt.tight_layout()
plt.show()


## 3. Tire Degradation EDA

In [ ]:
sample_stints = (
    eda_df[eda_df['Compound'].isin(['SOFT', 'MEDIUM', 'HARD'])]
    .groupby(['event_name', 'Compound', 'driver_stint_id'], as_index=False)['tire_age_laps']
    .max()
    .sort_values('tire_age_laps', ascending=False)
    .groupby(['event_name', 'Compound'])
    .head(1)['driver_stint_id']
)
plot_df = eda_df[eda_df['driver_stint_id'].isin(sample_stints)].copy()
g = sns.FacetGrid(plot_df, col='event_name', hue='Compound', col_wrap=3, sharey=False, height=3.4, palette={'SOFT':'#FF4444','MEDIUM':'#FFD700','HARD':'#888888'})
g.map_dataframe(sns.lineplot, x='tire_age_laps', y='lap_time_delta_s')
g.add_legend()
g.fig.suptitle('Representative Real-World Degradation Curves', y=1.02, fontsize=16, fontweight='bold')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=eda_df, x='Compound', y='deg_velocity', palette=['#888888', '#FFD700', '#FF4444'], ax=axes[0])
axes[0].set_title('Degradation Velocity by Compound')
sns.boxplot(data=eda_df, x='Compound', y='deg_acceleration', palette=['#888888', '#FFD700', '#FF4444'], ax=axes[1])
axes[1].set_title('Degradation Acceleration by Compound')
plt.tight_layout()
plt.show()


## 4. Correlation and Feature Relationships

In [ ]:
corr_cols = [
    'lap_time_delta_s', 'deg_velocity', 'deg_acceleration', 'tire_age_laps', 'tire_age_sq',
    'fuel_load_kg', 'gap_ahead_s', 'gap_ahead_trend', 'undercut_threat', 'safety_car_flag', 'lap_norm'
]
corr_df = eda_df[corr_cols + ['pit_label']].dropna().copy()
corr = corr_df.corr(method='spearman')
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Spearman Correlation Heatmap (Real Data)', fontsize=14, fontweight='bold')
plt.show()


## 5. Statistical Feature Checks

In [ ]:
feature_tests = []
for feature in ['deg_acceleration', 'deg_velocity', 'gap_ahead_s', 'gap_ahead_trend', 'fuel_load_kg', 'tire_age_laps']:
    groups = [grp[feature].dropna().values for _, grp in eda_df.groupby('pit_label')]
    groups = [g for g in groups if len(g) > 3]
    if len(groups) >= 2:
        anova_p = f_oneway(*groups).pvalue
        kruskal_p = kruskal(*groups).pvalue
        feature_tests.append({'feature': feature, 'anova_p': anova_p, 'kruskal_p': kruskal_p})

test_df = pd.DataFrame(feature_tests).sort_values('kruskal_p')
display(test_df)

pit_rate = eda_df.groupby('undercut_threat')['pit_label'].apply(lambda s: (s == 2).mean())
plt.figure(figsize=(6, 4))
pit_rate.plot(kind='bar', color=['#4A90D9', '#E10600'])
plt.title('Pit-Now Rate vs Undercut Threat')
plt.ylabel('Pit-Now rate')
plt.xticks([0, 1], ['No threat', 'Threat'], rotation=0)
plt.show()


## 6. Temporal Window Justification

In [ ]:
max_lag = 15
lag_results = []
for lag in range(1, max_lag + 1):
    per_stint = []
    for _, stint_df in eda_df.groupby('driver_stint_id'):
        series = stint_df.sort_values('LapNumber')['lap_time_delta_s'].dropna()
        if len(series) > lag + 2:
            per_stint.append(series.autocorr(lag=lag))
    lag_results.append({'lag': lag, 'mean_autocorr': np.nanmean(per_stint) if per_stint else np.nan})

lag_df = pd.DataFrame(lag_results)
plt.figure(figsize=(10, 4))
sns.barplot(data=lag_df, x='lag', y='mean_autocorr', color='#4A90D9')
plt.axvline(9.5, color='#E10600', linestyle='--', linewidth=2)
plt.title('Mean Autocorrelation by Lag for lap_time_delta_s')
plt.ylabel('Mean autocorrelation')
plt.show()


## 7. Circuit Fingerprint Analysis

In [ ]:
fingerprints = []
for circuit_name, circuit_df in eda_df.groupby('event_name'):
    slopes = []
    for compound in ['SOFT', 'MEDIUM', 'HARD']:
        sub = circuit_df[circuit_df['Compound'] == compound].dropna(subset=['tire_age_laps', 'lap_time_delta_s'])
        if len(sub) > 5:
            slopes.append(linregress(sub['tire_age_laps'], sub['lap_time_delta_s']).slope)
    fingerprints.append({
        'event_name': circuit_name,
        'avg_deg_rate': float(np.mean(slopes)) if slopes else 0.0,
        'compound_variance': float(circuit_df.groupby('Compound')['lap_time_delta_s'].mean().std() or 0.0),
        'track_temp_mean': float(circuit_df['AirTemp'].mean()),
        'gap_mean': float(circuit_df['gap_ahead_s'].mean()),
        'hist_soft_stint_len': float(circuit_df[circuit_df['Compound'] == 'SOFT'].groupby('driver_stint_id')['tire_age_laps'].max().median() or 0.0),
        'sc_vsc_frequency': float(circuit_df.groupby(['season', 'event_name'])['sc_or_vsc_flag'].max().mean()),
    })

fp_df = pd.DataFrame(fingerprints).sort_values('event_name')
display(fp_df)

fp_features = ['avg_deg_rate', 'compound_variance', 'track_temp_mean', 'gap_mean', 'hist_soft_stint_len', 'sc_vsc_frequency']
scaled = StandardScaler().fit_transform(fp_df[fp_features].fillna(0.0))
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(scaled)
fp_df['pc1'] = coords[:, 0]
fp_df['pc2'] = coords[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=fp_df, x='pc1', y='pc2', s=120, color='#E10600')
for _, row in fp_df.iterrows():
    plt.text(row['pc1'] + 0.03, row['pc2'] + 0.03, row['event_name'])
plt.title('PCA Projection of Real-World Circuit Fingerprints')
plt.show()


## 8. Constructor and Circuit Summary

In [ ]:
summary = pd.DataFrame({
    'metric': [
        'lap rows', 'stints', 'circuits', 'constructors', 'pit_now prevalence', 'pit_soon prevalence', 'stay_out prevalence'
    ],
    'value': [
        f'{len(eda_df):,}',
        f'{eda_df["driver_stint_id"].nunique():,}',
        ', '.join(sorted(eda_df['event_name'].unique())),
        ', '.join(sorted(eda_df['Team'].dropna().unique())),
        f'{(eda_df["pit_label"] == 2).mean() * 100:.2f}%',
        f'{(eda_df["pit_label"] == 1).mean() * 100:.2f}%',
        f'{(eda_df["pit_label"] == 0).mean() * 100:.2f}%',
    ]
})
display(summary)


## Notes

- This notebook is now backed by real FastF1 API data and cache.
- It is intentionally limited to the project v1 circuits and constructors.
- If Ergast/FastF1 rate limits are hit, rerun later after the cache window cools down.